# 10 - Prepare external MIMIC-IV-ECG waveform-diagnosis pairs

This notebook uses only `record_list.csv` and `machine_measurements.csv` from MIMIC-IV-ECG. It concatenates `report_0` through `report_17` into the machine diagnosis, resolves each WFDB record stem, removes blank reports, and makes deterministic patient-disjoint splits.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re

import pandas as pd

ARTIFACT_ROOT = Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts"))
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
ECG_ROOT = Path(config["external_ecg_root"])
OUT = ARTIFACT_ROOT / "external" / "ecg"
OUT.mkdir(parents=True, exist_ok=True)
RECORD_LIST = ECG_ROOT / "record_list.csv"
MACHINE = ECG_ROOT / "machine_measurements.csv"
for path in [RECORD_LIST, MACHINE]:
    if not path.is_file():
        raise FileNotFoundError(path)
print({"ecg_root": str(ECG_ROOT), "output": str(OUT)})


In [ ]:
record_list = pd.read_csv(RECORD_LIST, low_memory=False)
machine = pd.read_csv(MACHINE, low_memory=False)
required_record = {"subject_id", "study_id", "file_name"}
if missing := required_record - set(record_list.columns):
    raise ValueError(f"record_list.csv missing columns: {sorted(missing)}")
report_cols = sorted(
    [c for c in machine.columns if c.startswith("report_")],
    key=lambda value: int(value.split("_")[-1]),
)
if not report_cols:
    raise ValueError("machine_measurements.csv has no report_0...report_N columns")

def join_report(row):
    parts = []
    for col in report_cols:
        value = "" if pd.isna(row[col]) else str(row[col]).strip()
        if value and value.lower() != "nan" and value not in parts:
            parts.append(value)
    return " | ".join(parts)

machine = machine[["study_id", *report_cols]].copy()
machine["machine_diagnosis"] = machine.apply(join_report, axis=1)
machine = machine.loc[machine["machine_diagnosis"].str.len().gt(0), ["study_id", "machine_diagnosis"]]
machine = machine.drop_duplicates("study_id", keep="first")
linked = record_list.merge(machine, on="study_id", how="inner", validate="one_to_one")
print({"record_rows": len(record_list), "diagnosis_rows": len(machine), "linked_rows": len(linked)})


In [ ]:
RELEASE_PREFIX = "mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0"

# record_list.csv may contain an absolute path, files/p.../s.../study_id,
# p.../s.../study_id, or a path carrying the old release-directory prefix.
# Resolve the existing .hea file instead of assuming one extraction layout.
SEARCH_BASES = []
for candidate in [
    ECG_ROOT,
    ECG_ROOT / RELEASE_PREFIX,
    ECG_ROOT.parent,
    ECG_ROOT.parent / RELEASE_PREFIX,
    ECG_ROOT.parent.parent,
    ECG_ROOT.parent.parent / RELEASE_PREFIX,
]:
    candidate = candidate.resolve()
    if candidate not in SEARCH_BASES:
        SEARCH_BASES.append(candidate)

def without_wfdb_suffix(path):
    return path.with_suffix("") if path.suffix.lower() in {".hea", ".dat"} else path

def identifier_string(value):
    # CSV readers may represent an integer identifier as 40689238.0.
    try:
        return str(int(value))
    except (TypeError, ValueError, OverflowError):
        return str(value).strip()

def waveform_candidates(row):
    subject_id = identifier_string(row["subject_id"])
    study_id = identifier_string(row["study_id"])
    raw_value = identifier_string(row["file_name"])
    raw_path = without_wfdb_suffix(Path(raw_value))
    candidates = []
    if raw_path.is_absolute():
        candidates.append(raw_path)

    # Canonical MIMIC-IV-ECG layout, e.g.:
    # files/p1000/p10000032/s40689238/40689238
    patient_folder = f"p{subject_id}"
    patient_group = f"p{subject_id[:4]}"
    study_folder = f"s{study_id}"
    record_basename = raw_path.name
    canonical_relative = Path("files") / patient_group / patient_folder / study_folder / record_basename

    parts = raw_path.parts
    if "files" in parts:
        # Keep only the stable release-relative suffix: files/p.../s.../study_id.
        raw_relative = Path(*parts[parts.index("files"):])
    else:
        cleaned = [part for part in parts if part not in {RELEASE_PREFIX, "/"}]
        raw_relative = Path(*cleaned) if cleaned else raw_path
        if raw_relative.parts and raw_relative.parts[0].startswith("p"):
            raw_relative = Path("files") / raw_relative

    for base in SEARCH_BASES:
        # Try the canonical identifier-derived path first. The raw path is
        # retained as a fallback for releases whose file_name is already relative.
        candidates.append(without_wfdb_suffix(base / canonical_relative))
        candidates.append(without_wfdb_suffix(base / raw_relative))

    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique

def waveform_stem(row):
    candidates = waveform_candidates(row)
    for candidate in candidates:
        if candidate.with_suffix(".hea").is_file():
            return candidate
    # Return the preferred normalized location for a useful diagnostic below.
    return candidates[0]

def stable_split(subject_id):
    bucket = int(hashlib.sha1(str(int(subject_id)).encode()).hexdigest()[:8], 16) % 100
    return "train" if bucket < 80 else ("val" if bucket < 90 else "test")

linked["waveform_path"] = linked.apply(waveform_stem, axis=1).map(str)
linked["split"] = linked["subject_id"].map(stable_split)
linked["sample_id"] = linked["study_id"].map(lambda value: f"ecg_{int(value)}")
linked["waveform_exists"] = linked["waveform_path"].map(lambda value: Path(value).with_suffix(".hea").is_file())
missing_waveforms = int((~linked["waveform_exists"]).sum())
print("Resolved WFDB headers:", int(linked["waveform_exists"].sum()))
print("Missing WFDB headers:", missing_waveforms)

if linked["waveform_exists"].sum() == 0:
    examples = linked[["subject_id", "study_id", "file_name", "waveform_path"]].head(10).to_dict("records")
    searched = [str(path) for path in SEARCH_BASES]
    raise FileNotFoundError(
        "No WFDB .hea files were resolved. This caused the earlier split assertion. "
        f"ECG_ROOT={ECG_ROOT}; searched_bases={searched}; examples={examples}"
    )

manifest = linked.loc[linked["waveform_exists"], [
    "sample_id", "subject_id", "study_id", "waveform_path", "machine_diagnosis", "split"
]].copy()
if not manifest["sample_id"].is_unique:
    duplicates = manifest.loc[manifest["sample_id"].duplicated(False), "sample_id"].head(20).tolist()
    raise ValueError(f"Duplicate ECG sample_id values: {duplicates}")
split_counts = manifest.groupby("subject_id", dropna=False)["split"].nunique()
leaking_subjects = split_counts[split_counts > 1]
if not leaking_subjects.empty:
    raise AssertionError(f"Subjects assigned to multiple splits: {leaking_subjects.head(20).to_dict()}")
print(manifest.groupby("split").agg(rows=("sample_id", "size"), subjects=("subject_id", "nunique")))


In [ ]:
output = OUT / "manifest.jsonl"
with output.open("w") as stream:
    for row in manifest.to_dict("records"):
        stream.write(json.dumps(row) + "\n")
summary = {
    "rows": len(manifest), "subjects": int(manifest.subject_id.nunique()),
    "splits": manifest.split.value_counts().to_dict(),
    "source_record_list": str(RECORD_LIST), "source_machine_measurements": str(MACHINE),
    "missing_waveforms_removed": int((~linked["waveform_exists"]).sum()),
}
(OUT / "manifest_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved:", output)
